# 02 - Línea base clásica: Lexicones + SVM

Este notebook implementa el línea base clásica:
1. Extraer aspectos con reglas + POS tagging (spaCy).
2. Etiquetar sentimientos por aspecto usando un lexicón.
3. Entrenar un clasificador SVM TF-IDF con auxiliary sentence.

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("No se encontró la raíz del repositorio")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sklearn.model_selection import train_test_split

from src.aspects.extractor import extract_aspects, load_spacy_model
from src.classical.lexicon import DEFAULT_LEXICON, load_lexicon, score_aspect_lexicon
from src.classical.svm_classifier import SVMAspectClassifier
from src.data.loader import load_reviews
from src.data.builder import build_best_available_dataset
from src.evaluation.metrics import evaluate_predictions

## 1. Carga de datos y modelo spaCy

In [ ]:
df = load_reviews(REPO_ROOT / "data/sample/reviews_sample.csv")
nlp = load_spacy_model()
lexicon = load_lexicon()
print(f"Reseñas: {len(df)} | Lexicón: {len(lexicon)} términos")

## 2. Extracción de aspectos

Usamos POS tagging para sacar sustantivos relevantes.

In [ ]:
for _, row in df.head(3).iterrows():
    print('Reseña :', row['text'])
    print('Aspectos:', extract_aspects(row['text'], nlp))
    print('---')

## 3. Scoring por lexicón

Aplicamos `score_aspect_lexicon` con ventana de 5 palabras.

In [ ]:
sample = df.iloc[0]['text']
for asp in extract_aspects(sample, nlp):
    print(f"{asp:>15} -> {score_aspect_lexicon(sample, asp, lexicon):+.2f}")

## 4. Construcción del dataset

Si el CSV no trae anotaciones manuales, el constructor genera pseudo-etiquetas con el lexicón. Esas métricas son solo una prueba funcional.


In [ ]:
texts, aspects, labels, label_source = build_best_available_dataset(df)
print(f'Ejemplos: {len(texts)} | fuente de etiquetas: {label_source}')


## 5. Entrenamiento SVM

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['rating'] if df['rating'].value_counts().min() >= 2 else None,
)

t_tr, a_tr, y_tr, train_source = build_best_available_dataset(train_df)
t_te, a_te, y_te, test_source = build_best_available_dataset(test_df)
print(f'Train: {len(t_tr)} | Test: {len(t_te)} | etiquetas: {train_source}/{test_source}')

clf = SVMAspectClassifier()
clf.fit(t_tr, a_tr, y_tr)
preds = clf.predict(t_te, a_te).tolist()
metrics = evaluate_predictions(y_te, preds)
metrics


## 6. Conclusiones
- El lexicón sólo cubre términos comunes; aspectos sin palabras sentimentales cercanas devuelven 0.
- El SVM mejora sobre el lexicón si hay suficientes ejemplos.
- Comparar con BETO (notebook 03) y LLM (notebook 04).